
# Bank Marketing Optimization: Segmentation, Prediction, and Experiment Design

This notebook builds an end-to-end decision system for a term-deposit campaign: descriptive insights, customer segmentation, predictive targeting, leakage checks, and an experiment plan to validate business lift.



## 1. Business Problem

The bank wants to improve term-deposit conversion while reducing wasted outreach. Campaign calls are costly, and indiscriminate targeting lowers ROI.

**Business goals**
- Increase conversion rate from outbound campaigns.
- Reduce contact volume to low-propensity customers.
- Improve marketing ROI and cost per acquisition (CPA).

**Analytics goals**
- Segment customers into behaviorally distinct groups.
- Build an out-of-sample predictive model for subscription propensity.
- Separate descriptive signals from deployable pre-call signals (leakage control).


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import importlib.util
import subprocess
import sys
import time
from pathlib import Path

def ensure_package(module_name, pip_name=None):
    if importlib.util.find_spec(module_name) is None:
        pkg = pip_name or module_name
        print(f"Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

ensure_package("kagglehub")
ensure_package("mlxtend")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

pd.set_option("display.max_columns", 120)
RANDOM_STATE = 42


## 2. Dataset & EDA

We use the Bank Marketing dataset from Kaggle. The target is `deposit` (`yes`/`no`), indicating whether a customer subscribed to a term deposit.

EDA is focused on decision-relevant patterns: response distribution, feature quality, and variables likely to influence campaign success.


In [ ]:
# Load dataset with fallback strategy
df = None

# Option A: KaggleHub (preferred)
try:
    path = kagglehub.dataset_download("janiobachmann/bank-marketing-dataset")
    data_dir = Path(path)
    csv_candidates = sorted(data_dir.rglob("*.csv"))
    if csv_candidates:
        csv_path = csv_candidates[0]
        print("Using KaggleHub file:", csv_path)
        df = pd.read_csv(csv_path)
except Exception as e:
    print("KaggleHub load failed:", e)

# Option B: local fallback file names
if df is None:
    fallback_paths = [
        Path("bank.csv"),
        Path.cwd() / "bank.csv",
        Path("C:/Users/kk/Downloads/bank.csv")
    ]
    for p in fallback_paths:
        if p.exists():
            print("Using fallback local file:", p)
            df = pd.read_csv(p)
            break

if df is None:
    raise FileNotFoundError(
        "Dataset could not be loaded. Ensure internet access for KaggleHub or place bank.csv in the working directory."
    )

df.head()

In [ ]:
# Data quality and target checks
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes.value_counts())
print("\nMissing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

TARGET = "deposit"

target_counts = df[TARGET].value_counts()
print("\nTarget distribution:\n", target_counts)
print(f"Subscription rate: {(target_counts.get('yes', 0) / len(df)):.3f}")

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in df.select_dtypes(exclude=np.number).columns.tolist() if c != TARGET]
print("\nNumeric columns:", num_cols)
print("Categorical columns:", cat_cols)

In [ ]:
# Core EDA visuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target balance
(target_counts / len(df)).plot(kind="bar", ax=axes[0], color=["#4C78A8", "#F58518"])
axes[0].set_title("Target Distribution (Proportion)")
axes[0].set_xlabel("deposit")
axes[0].set_ylabel("proportion")
axes[0].tick_params(axis="x", rotation=0)

# Numeric correlation heatmap (matplotlib)
corr = df[num_cols].corr()
im = axes[1].imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_title("Numeric Feature Correlation")
axes[1].set_xticks(range(len(corr.columns)))
axes[1].set_yticks(range(len(corr.columns)))
axes[1].set_xticklabels(corr.columns, rotation=45, ha='right')
axes[1].set_yticklabels(corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        axes[1].text(j, i, f"{corr.values[i, j]:.2f}", ha='center', va='center', fontsize=8)

fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:

# Response-rate summary by key categorical dimensions
summary_frames = []
for col in cat_cols:
    rates = df.groupby(col)[TARGET].apply(lambda s: (s == "yes").mean()).sort_values(ascending=False)
    temp = rates.reset_index().rename(columns={TARGET: "yes_rate"})
    temp.insert(0, "feature", col)
    summary_frames.append(temp)

cat_response_summary = pd.concat(summary_frames, ignore_index=True)
cat_response_summary.head(20)


In [ ]:

# Numeric behavior by target
numeric_target_means = (
    df.groupby(TARGET)[num_cols]
      .mean()
      .T
      .rename(columns={"yes": "mean_yes", "no": "mean_no"})
)
numeric_target_means["delta_yes_minus_no"] = numeric_target_means["mean_yes"] - numeric_target_means["mean_no"]
numeric_target_means.sort_values("delta_yes_minus_no", ascending=False)



## 3. Customer Segmentation (KMeans)

Segmentation is used for campaign strategy design, not direct prediction. Clusters are profiled by behavior and conversion propensity.

Features are selected from numeric campaign/customer attributes. We keep `duration` for descriptive segmentation, but we later exclude it from deployable predictive modeling due to leakage risk.


In [ ]:

cluster_features = ["age", "balance", "duration", "campaign", "pdays", "previous"]
X_cluster = df[cluster_features].copy()

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

inertia = []
silhouette_scores = []
K_RANGE = range(2, 11)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cluster_scaled)
    inertia.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_RANGE), inertia, marker="o")
axes[0].set_title("Elbow Curve")
axes[0].set_xlabel("k")
axes[0].set_ylabel("inertia")

axes[1].plot(list(K_RANGE), silhouette_scores, marker="o")
axes[1].set_title("Silhouette by k")
axes[1].set_xlabel("k")
axes[1].set_ylabel("silhouette")

plt.tight_layout()
plt.show()

pd.DataFrame({"k": list(K_RANGE), "silhouette": silhouette_scores}).sort_values("silhouette", ascending=False)


In [ ]:

# Choose k=4 for actionable segmentation granularity
K_FINAL = 4
kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_cluster_scaled)

cluster_profile = df.groupby("cluster")[cluster_features].mean()
cluster_size = df["cluster"].value_counts().sort_index().rename("n_customers")
cluster_conversion = df.groupby("cluster")[TARGET].apply(lambda s: (s == "yes").mean()).rename("deposit_yes_rate")

cluster_summary = pd.concat([cluster_size, cluster_conversion, cluster_profile], axis=1).sort_index()
cluster_summary


In [ ]:

# PCA projection for cluster visualization
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["cluster"], cmap="tab10", s=15)
plt.title("KMeans Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()



## 4. Predictive Modeling (NEW)

We now build a deployable propensity model for `deposit`.

**Model set**
- Logistic Regression: interpretable baseline.
- Gradient Boosting (XGBoost-style boosting alternative): non-linear benchmark.

**Validation design**
- Stratified train/test split.
- Strict out-of-sample evaluation.


In [ ]:

# Prepare supervised learning dataset
X_all = df.drop(columns=[TARGET, "cluster"], errors="ignore").copy()
y = (df[TARGET] == "yes").astype(int)

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train size:", len(train_idx), "| Test size:", len(test_idx))
print("Train conversion rate:", y.loc[train_idx].mean().round(3))
print("Test conversion rate:", y.loc[test_idx].mean().round(3))

feature_sets = {
    "with_duration": X_all.columns.tolist(),
    "without_duration": [c for c in X_all.columns if c != "duration"]
}


def make_preprocessor(X_df, scale_numeric=True):
    num_features = X_df.select_dtypes(include=np.number).columns.tolist()
    cat_features = X_df.select_dtypes(exclude=np.number).columns.tolist()

    num_transformer = StandardScaler() if scale_numeric else "passthrough"

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_transformer, num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features)
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )
    return preprocessor


def evaluate_pipeline(pipe, X_train, X_test, y_train, y_test):
    pipe.fit(X_train, y_train)
    pred_proba = pipe.predict_proba(X_test)[:, 1]
    pred_label = (pred_proba >= 0.5).astype(int)

    metrics = {
        "roc_auc": roc_auc_score(y_test, pred_proba),
        "f1": f1_score(y_test, pred_label),
        "precision": precision_score(y_test, pred_label),
        "recall": recall_score(y_test, pred_label),
        "confusion_matrix": confusion_matrix(y_test, pred_label)
    }
    return metrics, pred_proba, pred_label



## 5. Model Evaluation (NEW)

We evaluate both models under two scenarios:
1. **With `duration`** (higher apparent accuracy but leakage-prone).
2. **Without `duration`** (deployable pre-call model).

Primary ranking metric is ROC-AUC, supported by F1, precision, recall, and confusion matrix.


In [ ]:

results = []
conf_mats = {}
fitted_models = {}

for fs_name, cols in feature_sets.items():
    X_train = X_all.loc[train_idx, cols]
    X_test = X_all.loc[test_idx, cols]
    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]

    lr_pipeline = Pipeline(steps=[
        ("preprocess", make_preprocessor(X_train, scale_numeric=True)),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ])

    gb_pipeline = Pipeline(steps=[
        ("preprocess", make_preprocessor(X_train, scale_numeric=False)),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])

    model_dict = {
        "Logistic Regression": lr_pipeline,
        "Gradient Boosting": gb_pipeline
    }

    for model_name, pipe in model_dict.items():
        metrics, pred_proba, pred_label = evaluate_pipeline(pipe, X_train, X_test, y_train, y_test)

        results.append({
            "feature_set": fs_name,
            "model": model_name,
            "roc_auc": metrics["roc_auc"],
            "f1": metrics["f1"],
            "precision": metrics["precision"],
            "recall": metrics["recall"]
        })

        conf_mats[(fs_name, model_name)] = metrics["confusion_matrix"]
        fitted_models[(fs_name, model_name)] = pipe

results_df = pd.DataFrame(results).sort_values(["feature_set", "roc_auc"], ascending=[True, False])
results_df


In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for ax, key in zip(axes, sorted(conf_mats.keys())):
    cm = conf_mats[key]
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{key[0]} | {key[1]}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')

plt.tight_layout()
plt.show()


**Evaluation interpretation**

- Compare `with_duration` vs `without_duration` first to understand leakage inflation.
- For deployment, prioritize the best `without_duration` model.
- Use threshold tuning later for business-specific trade-offs (maximize ROI, control contact volume).



## 6. Model Interpretation (NEW)

Interpretability is done on the **deployable** setup (`without_duration`) to ensure business actions are based on pre-call information.

- Logistic Regression: signed coefficients (positive/negative drivers).
- Gradient Boosting: feature importance ranking.


In [ ]:

# Extract deployable models
lr_deploy = fitted_models[("without_duration", "Logistic Regression")]
gb_deploy = fitted_models[("without_duration", "Gradient Boosting")]

# Logistic coefficients
lr_pre = lr_deploy.named_steps["preprocess"]
lr_model = lr_deploy.named_steps["model"]

feature_names_lr = lr_pre.get_feature_names_out()
coef_df = pd.DataFrame({
    "feature": feature_names_lr,
    "coefficient": lr_model.coef_.ravel()
}).sort_values("coefficient", ascending=False)

print("Top positive logistic drivers")
display(coef_df.head(15))

print("Top negative logistic drivers")
display(coef_df.tail(15).sort_values("coefficient", ascending=True))


In [ ]:
# Gradient boosting feature importances
gb_pre = gb_deploy.named_steps["preprocess"]
gb_model = gb_deploy.named_steps["model"]

feature_names_gb = gb_pre.get_feature_names_out()
importance_df = pd.DataFrame({
    "feature": feature_names_gb,
    "importance": gb_model.feature_importances_
}).sort_values("importance", ascending=False)

top_imp = importance_df.head(15).iloc[::-1]
plt.figure(figsize=(10, 6))
plt.barh(top_imp["feature"], top_imp["importance"], color="#4C78A8")
plt.title("Top 15 Gradient Boosting Feature Importances (No Duration)")
plt.xlabel("importance")
plt.ylabel("feature")
plt.tight_layout()
plt.show()

importance_df.head(15)

In [ ]:

# Practical targeting lift on test set (deployable best model)
deploy_results = results_df[results_df["feature_set"] == "without_duration"].sort_values("roc_auc", ascending=False)
best_deploy_model_name = deploy_results.iloc[0]["model"]
best_deploy_model = fitted_models[("without_duration", best_deploy_model_name)]

X_test_deploy = X_all.loc[test_idx, feature_sets["without_duration"]]
y_test_deploy = y.loc[test_idx].reset_index(drop=True)
score = best_deploy_model.predict_proba(X_test_deploy)[:, 1]

ranking_df = pd.DataFrame({"y_true": y_test_deploy, "score": score})

lift_rows = []
for pct in [0.10, 0.20, 0.30, 0.40, 0.50]:
    threshold = ranking_df["score"].quantile(1 - pct)
    top_slice = ranking_df[ranking_df["score"] >= threshold]
    lift_rows.append({
        "top_fraction": pct,
        "n_customers": len(top_slice),
        "conversion_rate": top_slice["y_true"].mean(),
        "lift_vs_baseline": top_slice["y_true"].mean() / ranking_df["y_true"].mean()
    })

lift_table = pd.DataFrame(lift_rows)
print(f"Best deployable model: {best_deploy_model_name}")
display(lift_table)



## 7. Leakage & Robustness Analysis (NEW)

### 7.1 Leakage check: `duration`

`duration` is only fully known after a call is in progress and is strongly tied to outcome. Using it for pre-call targeting inflates model quality and makes the model hard to deploy operationally.

We report both scenarios and use **without-duration** performance as the deployable benchmark.


In [ ]:

# Leakage impact summary
leakage_compare = results_df.pivot(index="model", columns="feature_set", values=["roc_auc", "f1", "precision", "recall"])
leakage_compare


In [ ]:

# Robustness via stratified cross-validated ROC-AUC (deployable feature set)
X_deploy = X_all[feature_sets["without_duration"]]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

cv_models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocess", make_preprocessor(X_deploy, scale_numeric=True)),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "Gradient Boosting": Pipeline(steps=[
        ("preprocess", make_preprocessor(X_deploy, scale_numeric=False)),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
}

for model_name, model in cv_models.items():
    auc_scores = cross_val_score(model, X_deploy, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_rows.append({
        "model": model_name,
        "cv_auc_mean": auc_scores.mean(),
        "cv_auc_std": auc_scores.std()
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values("cv_auc_mean", ascending=False)
cv_results_df



### 7.2 Actionable Association Rules (Improved)

We keep association rules only if they are usable for campaign decisions.

**Actionable rule criteria**
- Antecedent must not contain `deposit_*` (target leakage in rule body).
- Consequent must be `deposit_yes` (targeting objective).
- Rule must pass minimum support, confidence, and lift thresholds.

We also benchmark Apriori vs FP-Growth with identical settings and report runtime correctly.


In [ ]:

rule_cols = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "poutcome", "deposit"
]

df_rules = df[rule_cols].copy()
encoded = pd.get_dummies(df_rules, dtype=bool)

min_support = 0.05
min_confidence = 0.60
min_lift = 2.0

# Apriori runtime
start_ap = time.time()
fi_ap = apriori(encoded, min_support=min_support, use_colnames=True)
rules_ap = association_rules(fi_ap, metric="lift", min_threshold=min_lift)
rules_ap = rules_ap[rules_ap["confidence"] >= min_confidence].copy()
ap_time = time.time() - start_ap

# FP-Growth runtime
start_fp = time.time()
fi_fp = fpgrowth(encoded, min_support=min_support, use_colnames=True)
rules_fp = association_rules(fi_fp, metric="lift", min_threshold=min_lift)
rules_fp = rules_fp[rules_fp["confidence"] >= min_confidence].copy()
fp_time = time.time() - start_fp


def as_rule_set(rules_df):
    return set(
        (
            tuple(sorted(list(a))),
            tuple(sorted(list(c)))
        )
        for a, c in zip(rules_df["antecedents"], rules_df["consequents"])
    )

overlap_count = len(as_rule_set(rules_ap) & as_rule_set(rules_fp))

runtime_df = pd.DataFrame({
    "algorithm": ["Apriori", "FP-Growth"],
    "n_itemsets": [len(fi_ap), len(fi_fp)],
    "n_rules": [len(rules_ap), len(rules_fp)],
    "runtime_seconds": [ap_time, fp_time]
})

runtime_df


In [ ]:

# Filter for actionable rules only

def is_actionable(antecedents, consequents):
    ant_has_target = any(item.startswith("deposit_") for item in antecedents)
    cons_is_positive_target = consequents == frozenset({"deposit_yes"})
    return (not ant_has_target) and cons_is_positive_target

rules_source = rules_fp.copy()  # use FP-Growth output for faster generation
rules_source["actionable"] = rules_source.apply(
    lambda r: is_actionable(r["antecedents"], r["consequents"]),
    axis=1
)

actionable_rules = rules_source[rules_source["actionable"]].copy()
actionable_rules = actionable_rules.sort_values(["lift", "confidence", "support"], ascending=False)

print("Total rules before actionability filter:", len(rules_source))
print("Actionable rules after filter:", len(actionable_rules))
print("Rule overlap (Apriori vs FP-Growth):", overlap_count)
print("Faster algorithm this run:", runtime_df.sort_values("runtime_seconds").iloc[0]["algorithm"])

cols_to_show = ["antecedents", "consequents", "support", "confidence", "lift"]
display(actionable_rules[cols_to_show].head(15))



## 8. Business Recommendations

1. **Target high-propensity customers first**
- Use the best deployable model (without `duration`) to rank customers.
- Prioritize top-score bands (e.g., top 20-30%) where observed lift is highest.

2. **Segment-aware campaign strategy**
- Prioritize clusters with higher historical conversion.
- Reduce repeated outreach to over-contacted, low-conversion segments.

3. **Channel and data-quality improvement**
- `contact_unknown` and weak historical campaign metadata are repeatedly linked to poor outcomes.
- Improve channel completeness and routing quality before scaling volume.

4. **Operational optimization**
- Use probability thresholds tied to budget and call-center capacity.
- Monitor precision/recall trade-off weekly and recalibrate thresholds.



## 9. A/B Testing Plan (NEW)

### Experiment design
- **Control**: current/random targeting strategy.
- **Treatment**: model-based targeting using deployable propensity scores.

### Hypotheses
- **H0**: Treatment conversion rate <= Control conversion rate.
- **H1**: Treatment conversion rate > Control conversion rate.

### Primary metrics
- Conversion rate (`deposit_yes / contacted`).
- Cost per acquisition (CPA).
- Campaign ROI.

### Secondary guardrails
- Contact rate fairness across customer groups.
- Unsubscribe/complaint rates.
- Call-center load stability.


In [ ]:

# Back-of-the-envelope sample size and ROI scenario
baseline_cr = y.mean()          # baseline conversion from historical data
expected_uplift = 0.03          # absolute uplift target (e.g., +3pp)
alpha = 0.05
power = 0.80

# z-scores for two-sided alpha=0.05 and power=0.80
z_alpha = 1.96
z_beta = 0.84

p1 = baseline_cr
p2 = baseline_cr + expected_uplift
p_bar = (p1 + p2) / 2

n_per_group = int(np.ceil((2 * (z_alpha + z_beta) ** 2 * p_bar * (1 - p_bar)) / (expected_uplift ** 2)))

print(f"Baseline conversion: {baseline_cr:.3f}")
print(f"Target uplift (absolute): {expected_uplift:.3f}")
print(f"Estimated minimum sample size per arm: {n_per_group:,}")

# Simple ROI illustration
contact_cost = 2.0
value_per_conversion = 120.0

roi_control = ((p1 * value_per_conversion) - contact_cost) / contact_cost
roi_treatment = ((p2 * value_per_conversion) - contact_cost) / contact_cost

print(f"Illustrative ROI index (control): {roi_control:.2f}")
print(f"Illustrative ROI index (treatment): {roi_treatment:.2f}")
print(f"Illustrative ROI lift index: {roi_treatment - roi_control:.2f}")
